# Analisis EDA

### Importacion de librerias: 
Aqui se cargan las librerias necesarias para el análisis exploratorio de datos (EDA).

In [7]:
import pandas as pd
import numpy as np
import yaml
import datetime
import matplotlib.pyplot as plt
import os
import re


In [2]:

# carga de configuración
with open("../config/config.yaml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

# cambiar el directorio de trabajo al nivel superior del proyecto
os.chdir("../")


### Descripción de los datos y transformaciones - tabla proyecciones_poblacion_dane(1)

In [3]:
# cargar datos
# obligar a leer como texto las columnas especificadas
columns = ['DP', 'DPNOM', 'MPIO', 'DPMP', 'AÑO', 'ÁREA GEOGRÁFICA']
df_proyecciones_poblacion_dane = pd.read_excel(config["sources"]["proyecciones_poblacion_dane"]["path"], sheet_name=config["sources"]["proyecciones_poblacion_dane"]["sheetname"], dtype={col: str for col in columns})


# df_proyecciones_poblacion_dane = pd.read_excel(config["sources"]["proyecciones_poblacion_dane"]["path"], sheet_name=config["sources"]["proyecciones_poblacion_dane"]["sheetname"])

In [4]:
df_proyecciones_poblacion_dane.head()

,DP,DPNOM,MPIO,DPMP,AÑO,ÁREA GEOGRÁFICA,Total,Hombres,Mujeres,Hombres 0 años,...,Total 91 años,Total 92 años,Total 93 años,Total 94 años,Total 95 años,Total 96 años,Total 97 años,Total 98 años,Total 99 años,Total 100 años y más
0,05,Antioquia,05001,Medellín,2018,Cabecera Municipal,2382405,1118868,1263537,13512,...,2027,1624,1282,1001,775,599,458,331,223,798
1,05,Antioquia,05001,Medellín,2018,Centros Poblados y Rural Disperso,44728,22475,22253,304,...,25,20,16,12,9,7,5,3,1,9
2,05,Antioquia,05001,Medellín,2018,Total,2427133,1141343,1285790,13816,...,2052,1644,1298,1013,784,606,463,334,224,807
3,05,Antioquia,05002,Abejorral,2018,Cabecera Municipal,7669,3646,4023,38,...,8,6,5,5,3,2,2,0,1,8
4,05,Antioquia,05002,Abejorral,2018,Centros Poblados y Rural Disperso,12699,6848,5851,96,...,6,4,3,2,1,1,2,1,0,2


In [8]:
#renombrar columnas de acuerdo a un listado
columns_rename = {
    'DP': 'COD_DPTO',
    'DPNOM': 'NOM_DPTO',
    'MPIO': 'COD_MPIO',
    'DPMP': 'NOM_MPIO',
    'AÑO': 'ANIO',
    'total': 'TOTAL_POBLACION',
    'Hombres': 'TOTAL_HOMBRES',
    'Mujeres': 'TOTAL_MUJERES',
    
}

df_proyecciones_poblacion_dane.rename(columns=columns_rename)

,COD_DPTO,NOM_DPTO,COD_MPIO,NOM_MPIO,ANIO,ÁREA GEOGRÁFICA,Total,TOTAL_HOMBRES,TOTAL_MUJERES,Hombres 0 años,...,Total 91 años,Total 92 años,Total 93 años,Total 94 años,Total 95 años,Total 96 años,Total 97 años,Total 98 años,Total 99 años,Total 100 años y más
0,05,Antioquia,05001,Medellín,2018,Cabecera Municipal,2382405,1118868,1263537,13512,...,2027,1624,1282,1001,775,599,458,331,223,798
1,05,Antioquia,05001,Medellín,2018,Centros Poblados y Rural Disperso,44728,22475,22253,304,...,25,20,16,12,9,7,5,3,1,9
2,05,Antioquia,05001,Medellín,2018,Total,2427133,1141343,1285790,13816,...,2052,1644,1298,1013,784,606,463,334,224,807
3,05,Antioquia,05002,Abejorral,2018,Cabecera Municipal,7669,3646,4023,38,...,8,6,5,5,3,2,2,0,1,8
4,05,Antioquia,05002,Abejorral,2018,Centros Poblados y Rural Disperso,12699,6848,5851,96,...,6,4,3,2,1,1,2,1,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84220,99,Vichada,99624,Santa Rosalía,2042,Centros Poblados y Rural Disperso,1526,834,692,8,...,3,2,3,1,2,1,1,0,2,1
84221,99,Vichada,99624,Santa Rosalía,2042,Total,3716,1914,1802,19,...,6,5,5,4,5,3,3,2,3,1
84222,99,Vichada,99773,Cumaribo,2042,Cabecera Municipal,4461,2194,2267,28,...,3,3,3,3,2,2,1,2,1,1
84223,99,Vichada,99773,Cumaribo,2042,Centros Poblados y Rural Disperso,176148,93145,83003,1383,...,22,15,12,8,5,4,2,2,0,13


In [9]:
# calcular poblacion por sexo y rango de edad (cada columna trae sexo y edad en el nombre)
# \s+ porque 'Total 100 años  y más' viene con doble espacio en la fuente
patron_edad = r"^(Hombres|Mujeres|Total) (\d+) años?(?:\s+y más)?$"

def cols_edad(sexo, edad_min, edad_max):
    return [c for c in df_proyecciones_poblacion_dane.columns
            if (m := re.match(patron_edad, c)) and m.group(1) == sexo and edad_min <= int(m.group(2)) <= edad_max]

df_proyecciones_poblacion_dane["HOMBRES_65_MAS"] = df_proyecciones_poblacion_dane[cols_edad("Hombres", 65, 100)].sum(axis=1)
df_proyecciones_poblacion_dane["MUJERES_65_MAS"] = df_proyecciones_poblacion_dane[cols_edad("Mujeres", 65, 100)].sum(axis=1)
df_proyecciones_poblacion_dane["HOMBRES_0_15"] = df_proyecciones_poblacion_dane[cols_edad("Hombres", 0, 15)].sum(axis=1)
df_proyecciones_poblacion_dane["MUJERES_0_15"] = df_proyecciones_poblacion_dane[cols_edad("Mujeres", 0, 15)].sum(axis=1)

df_proyecciones_poblacion_dane["TOTAL_65_MAS"] = df_proyecciones_poblacion_dane["HOMBRES_65_MAS"] + df_proyecciones_poblacion_dane["MUJERES_65_MAS"]
df_proyecciones_poblacion_dane["TOTAL_0_15"] = df_proyecciones_poblacion_dane["HOMBRES_0_15"] + df_proyecciones_poblacion_dane["MUJERES_0_15"]

# eliminar el detalle por edad (0 a 100 y más) de hombres, mujeres y total
columnas_detalle_edad = [c for c in df_proyecciones_poblacion_dane.columns if re.match(patron_edad, c)]
df_proyecciones_poblacion_dane = df_proyecciones_poblacion_dane.drop(columns=columnas_detalle_edad)

df_proyecciones_poblacion_dane.head()

C:\Users\juanc\AppData\Local\Temp\ipykernel_28372\1844214752.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_proyecciones_poblacion_dane["HOMBRES_65_MAS"] = df_proyecciones_poblacion_dane[cols_edad("Hombres", 65, 100)].sum(axis=1)
C:\Users\juanc\AppData\Local\Temp\ipykernel_28372\1844214752.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_proyecciones_poblacion_dane["MUJERES_65_MAS"] = df_proyecciones_poblacion_dane[cols_edad("Mujeres", 65, 100)].sum(axis=1)
C:\Users\juanc\AppData\Local\Temp\ipykernel_28372\1844214

,DP,DPNOM,MPIO,DPMP,AÑO,ÁREA GEOGRÁFICA,Total,Hombres,Mujeres,HOMBRES_65_MAS,MUJERES_65_MAS,HOMBRES_0_15,MUJERES_0_15,TOTAL_65_MAS,TOTAL_0_15
0,05,Antioquia,05001,Medellín,2018,Cabecera Municipal,2382405,1118868,1263537,100754,156034,253011,243872,256788,496883
1,05,Antioquia,05001,Medellín,2018,Centros Poblados y Rural Disperso,44728,22475,22253,2039,2112,5361,5109,4151,10470
2,05,Antioquia,05001,Medellín,2018,Total,2427133,1141343,1285790,102793,158146,258372,248981,260939,507353
3,05,Antioquia,05002,Abejorral,2018,Cabecera Municipal,7669,3646,4023,526,694,875,820,1220,1695
4,05,Antioquia,05002,Abejorral,2018,Centros Poblados y Rural Disperso,12699,6848,5851,761,561,1843,1765,1322,3608


### Análisis exploratorio - tabla relacion_ips_capacidad_instalada (2)

In [10]:

# cargar datos de proyecciones de 2.relacion_ips_capacidad_instalada con utf-8
df_relacion_ips_capacidad_instalada = pd.read_csv(
    config["sources"]["relacion_ips_capacidad_instalada"]["path"],
    sep=';',  # Cambia a punto y coma si usa eso
    encoding='utf-8',
    on_bad_lines='skip'  # Salta líneas problemáticas
)
columnas_texto = ['Departamento','naturaleza','nom descripcion capacidad ']
# Convertir columnas de texto en mayúsculasc con un for 
for col in columnas_texto:
    df_relacion_ips_capacidad_instalada[col] = df_relacion_ips_capacidad_instalada[col].apply(lambda x: x.upper() if isinstance(x, str) else x)

In [11]:
df_relacion_ips_capacidad_instalada.head()

,Departamento,Municipio,Código prestador,Nombre prestador,nit IPS,num digito_verificion,naturaleza,num nivel atencion,Código sede,Número sede,nom sede IPS,Gerente,Dirección,Email,Teléfono,nom grupo capacidad,nom descripcion capacidad,num cantidad capacidad instalada,Fecha Corte,Fuente
0,ANTIOQUIA,APARTADÓ,504512253,URAMEDICOS S.A.S.,"900,497,151",1.0,PRIVADA,NaN,504512253,1,URAMEDICOS,CLAUDIA CECILIA TRUJILLO GOMEZ,KR 98 # 103-29/37,uramedicos@gmail.com,8286090,SALAS,PROCEDIMIENTOS,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
1,ANTIOQUIA,SALGAR,564204576,ESE HOSPITAL SAN JOSE,"890,981,424",7.0,PÚBLICA,1.0,564204576,1,E.S.E. HOSPITAL SAN JOSE,CESAR AUGUSTO PARDO ROSAS,CL 32A # 33-04,secretaria@hsjsalgar.gov.co,8442020,CAMAS,TPR,2.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
2,AMAZONAS,EL ENCANTO,9100100019,E.S.E. HOSPITAL SAN RAFAEL DE LETICIA,"838,000,096",7.0,PÚBLICA,2.0,9126300019,11,CENTRO DE SALUD SAN RAFAEL - E.S.E HOSPITAL SA...,SAIDA VIVIANA HERREÑO PRIETO,CORREGIMIENTO DEL ENCANTO,sanrafael@esehospitalsanrafael-leticia-amazona...,3203016139,CAMAS,PEDIÁTRICA,3.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
3,AMAZONAS,EL ENCANTO,9100100019,E.S.E. HOSPITAL SAN RAFAEL DE LETICIA,"838,000,096",7.0,PÚBLICA,2.0,9126300019,11,CENTRO DE SALUD SAN RAFAEL - E.S.E HOSPITAL SA...,SAIDA VIVIANA HERREÑO PRIETO,CORREGIMIENTO DEL ENCANTO,sanrafael@esehospitalsanrafael-leticia-amazona...,3203016139,CAMAS,ADULTOS,4.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
4,AMAZONAS,EL ENCANTO,9100100019,E.S.E. HOSPITAL SAN RAFAEL DE LETICIA,"838,000,096",7.0,PÚBLICA,2.0,9126300019,11,CENTRO DE SALUD SAN RAFAEL - E.S.E HOSPITAL SA...,SAIDA VIVIANA HERREÑO PRIETO,CORREGIMIENTO DEL ENCANTO,sanrafael@esehospitalsanrafael-leticia-amazona...,3203016139,CAMAS,TPR,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...


### Análisis exploratorio tabla Divipola (4)

In [12]:
# lectura de tabla divipola, forzar a leer todas las columnas en texto
df_divipola = pd.read_excel(config["sources"]["divipola"]["path"], sheet_name=config["sources"]["divipola"]["sheetname"], dtype=str)

## TRANSFORMACIÓN DE LA TABLA 2 RELACION_IPS_CAPACIDAD_INSTALADA

#### Debido a que la tabla 2. relacion_ips_capacidad_instalada no tiene codigo de departamento ni de municipio pero si tiene los nombres tanto de departamento como municipio, se pueden traer mediante merge para asi poder tener los campos llave correspondientes cruzandolo con la tabla 4.DIVIPOLA

In [14]:
df_capacidad_instalada=df_relacion_ips_capacidad_instalada.copy()

# quitar tildes a las columnas seleccionadas
df_capacidad_instalada[['Departamento','Municipio']] = df_capacidad_instalada[['Departamento','Municipio']].apply(lambda x: x.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8'))


In [15]:
df_dim_divipola=df_divipola.copy()

# quitar tildes a las columnas seleccionadas
df_dim_divipola[['NOMBRE_DPTO','NOMBRE_MUN']] = df_dim_divipola[['NOMBRE_DPTO','NOMBRE_MUN']].apply(lambda x: x.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8'))


In [16]:
# Comparar NOMBRE_DPTO con Departamento
departamentos_divipola = set(df_dim_divipola['NOMBRE_DPTO'].unique())
departamentos_ips = set(df_capacidad_instalada['Departamento'].unique())

print("Departamentos en DIVIPOLA pero NO en IPS:")
print(departamentos_divipola - departamentos_ips)
print("\nDepartamentos en IPS pero NO en DIVIPOLA:")
print(departamentos_ips - departamentos_divipola)
print("\nDepartamentos que SÍ coinciden:")
print(departamentos_divipola & departamentos_ips)


Departamentos en DIVIPOLA pero NO en IPS:
{nan, 'BOGOTA, D.C.', 'ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA'}

Departamentos en IPS pero NO en DIVIPOLA:
{'BOGOTA D.C', 'CALI', 'CARTAGENA', 'BUENAVENTURA', 'SAN ANDRES Y PROVIDENCIA', 'SANTA MARTA', 'BARRANQUILLA'}

Departamentos que SÍ coinciden:
{'MAGDALENA', 'CAUCA', 'CESAR', 'GUAVIARE', 'HUILA', 'BOYACA', 'AMAZONAS', 'NARINO', 'CUNDINAMARCA', 'CASANARE', 'VALLE DEL CAUCA', 'VAUPES', 'CALDAS', 'ANTIOQUIA', 'LA GUAJIRA', 'ATLANTICO', 'GUAINIA', 'CORDOBA', 'QUINDIO', 'CAQUETA', 'SUCRE', 'BOLIVAR', 'ARAUCA', 'CHOCO', 'SANTANDER', 'TOLIMA', 'META', 'NORTE DE SANTANDER', 'PUTUMAYO', 'RISARALDA', 'VICHADA'}


In [17]:
## correciones de los departamentos que estan en IPS y no en divipola
valores_cambiar = {'BUENAVENTURA':'VALLE DEL CAUCA',
                   'CALI':'VALLE DEL CAUCA',
                   'BARRANQUILLA':'ATLANTICO',
                   'BOGOTA D.C':'BOGOTA, D.C.',
                   'SANTA MARTA':'MAGDALENA',
                   'CARTAGENA':'BOLIVAR',
                   'SAN ANDRES Y PROVIDENCIA':'ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA'
                  }

# Aplicar las correcciones a la columna de departamentos en la tabla df_dim_sede_ips
df_capacidad_instalada['Departamento'] = df_capacidad_instalada['Departamento'].replace(valores_cambiar)

In [18]:
# validacion de los reemplazos
departamentos_divipola = set(df_dim_divipola['NOMBRE_DPTO'].unique())
departamentos_ips = set(df_capacidad_instalada['Departamento'].unique())

print("Departamentos en DIVIPOLA pero NO en IPS:")
print(departamentos_divipola - departamentos_ips)
print("\nDepartamentos en IPS pero NO en DIVIPOLA:")
print(departamentos_ips - departamentos_divipola)
print("\nDepartamentos que SÍ coinciden:")
print(departamentos_divipola & departamentos_ips)


Departamentos en DIVIPOLA pero NO en IPS:
{nan}

Departamentos en IPS pero NO en DIVIPOLA:
set()

Departamentos que SÍ coinciden:
{'MAGDALENA', 'CAUCA', 'CESAR', 'GUAVIARE', 'ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA', 'HUILA', 'BOYACA', 'AMAZONAS', 'NARINO', 'CUNDINAMARCA', 'CASANARE', 'VALLE DEL CAUCA', 'VAUPES', 'CALDAS', 'ANTIOQUIA', 'LA GUAJIRA', 'ATLANTICO', 'GUAINIA', 'CORDOBA', 'QUINDIO', 'CAQUETA', 'SUCRE', 'BOLIVAR', 'ARAUCA', 'CHOCO', 'SANTANDER', 'TOLIMA', 'META', 'NORTE DE SANTANDER', 'PUTUMAYO', 'RISARALDA', 'BOGOTA, D.C.', 'VICHADA'}


In [19]:
# Comparar NOMBRE_MUN con Municipio 

municipios_divipola = set(df_dim_divipola['NOMBRE_MUN'].unique())
municipios_ips = set(df_capacidad_instalada['Municipio'].unique())

print("Municipios en DIVIPOLA pero NO en IPS:")
print(municipios_divipola - municipios_ips)
print("\nMunicipios en IPS pero NO en DIVIPOLA:")
print(municipios_ips - municipios_divipola)
print(f"\nTotal municipios que coinciden: {len(municipios_divipola & municipios_ips)}")


Municipios en DIVIPOLA pero NO en IPS:
{'PAPUNAHUA', 'BARRANCOMINAS', 'SOTARA PAISPAMBA', 'MORICHAL', 'SAN ANDRES DE CUERQUIA', 'SAN PEDRO DE LOS MILAGROS', 'SAN ANDRES DE SOTAVENTO', 'CUASPUD CARLOSAMA', 'SAN VICENTE FERRER', nan, 'SANTA CRUZ DE MOMPOX', 'CACAHUAL', 'SANTA FE DE ANTIOQUIA', 'SAN JUAN DE RIOSECO', 'PACOA', 'YAVARATE', 'GUICAN DE LA SIERRA', 'SAN JOSE DE CUCUTA', 'SAN ANDRES DE TUMACO', 'NUEVO BELEN DE BAJIRA', 'SAN SEBASTIAN DE MARIQUITA', 'CHIVOLO', 'SAN JOSE DE TOLUVIEJO', 'PURISIMA DE LA CONCEPCION', 'PANA PANA', 'LOPEZ DE MICAY', 'MANAURE BALCON DEL CESAR', 'PUERTO LEGUIZAMO', 'CARTAGENA DE INDIAS', 'LA GUADALUPE', 'SAN LUIS DE SINCE', 'CERRO DE SAN ANTONIO', 'PIENDAMO - TUNIA', 'SANTIAGO DE CALI', 'DONMATIAS', 'SOTARA ', 'BOGOTA, D.C.'}

Municipios en IPS pero NO en DIVIPOLA:
{'TUMACO', 'SAN ANDRES SOTAVENTO', 'SINCE', 'PIENDAMO', 'CHIBOLO', 'BARRANCO MINAS', 'LEGUIZAMO', 'CALI', 'TOLU VIEJO', 'DON MATIAS', 'CUASPUD', 'MOMPOS', 'LOPEZ', 'CUCUTA', 'SANTAFE DE ANTIO

In [21]:
# para Municipios en IPS pero NO en DIVIPOLA traer los registros correspondientes
municipios_no_en_divipola = municipios_ips - municipios_divipola
df_municipios_no_en_divipola = df_capacidad_instalada[df_capacidad_instalada['Municipio'].isin(municipios_no_en_divipola)]
df_municipios_no_en_divipola.head()

,Departamento,Municipio,Código prestador,Nombre prestador,nit IPS,num digito_verificion,naturaleza,num nivel atencion,Código sede,Número sede,nom sede IPS,Gerente,Dirección,Email,Teléfono,nom grupo capacidad,nom descripcion capacidad,num cantidad capacidad instalada,Fecha Corte,Fuente
488,"BOGOTA, D.C.",BOGOTA,1100137431,AMBUSURE IPS S.A.S.,"901,457,751",3.0,PRIVADA,NaN,1100137431,1,AMBUSURE IPS,OSCAR ALONSO ROJAS VARGAS,TR 77 C 47-16 P1,ambusureips@gmail.com,3224882104,AMBULANCIAS,BÁSICA,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
851,"BOGOTA, D.C.",BOGOTA,1100137431,AMBUSURE IPS S.A.S.,"901,457,751",3.0,PRIVADA,NaN,1100137431,1,AMBUSURE IPS,OSCAR ALONSO ROJAS VARGAS,TR 77 C 47-16 P1,ambusureips@gmail.com,3224882104,AMBULANCIAS,BÁSICA,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
950,ANTIOQUIA,DON MATIAS,523704806,COOPERATIVA DE TRABAJO ASOCIADO DE PROFESIONAL...,"800,228,773",7.0,PRIVADA,NaN,523704806,4,PROSALCO DON MATIAS,HERNAN DARIO VALLEJO RESTREPO,CL 29 # 28-55,asisgerencia@prosalcoips.com.co,4038424 ext 1123,CONSULTORIOS,CONSULTA EXTERNA,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
951,ANTIOQUIA,DON MATIAS,523704806,COOPERATIVA DE TRABAJO ASOCIADO DE PROFESIONAL...,"800,228,773",7.0,PRIVADA,NaN,523704806,4,PROSALCO DON MATIAS,HERNAN DARIO VALLEJO RESTREPO,CL 29 # 28-55,asisgerencia@prosalcoips.com.co,4038424 ext 1123,SALAS,PROCEDIMIENTOS,3.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
952,ANTIOQUIA,DON MATIAS,523705948,ESE HOSPITAL FRANCISCO ELADIO BARRERA,"890,905,097",8.0,PÚBLICA,1.0,523705948,1,E.S.E. HOSPITAL FRANCISCO ELADIO BARRERA,OSCAR HERNÁN OCAMPO VARELA,CL 36A # 29-55,subgerencia@hospitaldonmatias.gov.co,8663737,AMBULANCIAS,BÁSICA,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...


In [22]:
df_dim_divipola.head()

,COD_DPTO,NOMBRE_DPTO,COD_MUN,NOMBRE_MUN,COD_CENTRO_POBLADO,NOMBRE_CENTRO_POBLADO,TIPO_CENTRO_POBLADO,LONG,LAT
0,05,ANTIOQUIA,05001,MEDELLIN,05001000,"MEDELLÍN, DISTRITO ESPECIAL DE CIENCIA, TECNOL...",CM,-75.581775,6.246631
1,05,ANTIOQUIA,05001,MEDELLIN,05001001,PALMITAS,CP,-75.690573,6.343919
2,05,ANTIOQUIA,05001,MEDELLIN,05001004,SANTA ELENA,CP,-75.501293,6.210599
3,05,ANTIOQUIA,05001,MEDELLIN,05001009,ALTAVISTA,CP,-75.643706,6.221429
4,05,ANTIOQUIA,05001,MEDELLIN,05001010,AGUAS FRÍAS,CP,-75.633948,6.233335


In [23]:
## correciones de los municipios en la tabla df_dim_sede_ips
valores_cambiar = {'BARRANCO MINAS': 'BARRANCOMINAS',
                   'LEGUIZAMO': 'PUERTO LEGUIZAMO',
                   'LOPEZ': 'LOPEZ DE MICAY',
                   'BOGOTA': 'BOGOTA, D.C.',
                   'PURISIMA': 'PURISIMA DE LA CONCEPCION',
                   'MOMPOS': 'SANTA CRUZ DE MOMPOX',
                   'CALI': 'SANTIAGO DE CALI',
                   'SINCE': 'SAN LUIS DE SINCE',
                   'SAN VICENTE': 'SAN VICENTE FERRER',
                   'SAN ANDRES SOTAVENTO': 'SAN ANDRES DE SOTAVENTO',
                   'TOLU VIEJO': 'SAN JOSE DE TOLUVIEJO',
                   'CUASPUD': 'CUASPUD CARLOSAMA',
                   'CARTAGENA': 'CARTAGENA DE INDIAS',
                   'GUICAN': 'GUICAN DE LA SIERRA',
                   'SANTAFE DE ANTIOQUIA': 'SANTA FE DE ANTIOQUIA',
                   'CERRO SAN ANTONIO': 'CERRO DE SAN ANTONIO',
                   'DON MATIAS': 'DONMATIAS',
                   'CHIBOLO': 'CHIVOLO',
                   'SAN JUAN DE RIO SECO': 'SAN JUAN DE RIOSECO',
                   'PIENDAMO': 'PIENDAMO - TUNIA',
                   'MARIQUITA': 'SAN SEBASTIAN DE MARIQUITA',
                   'CUCUTA': 'SAN JOSE DE CUCUTA',
                   'SOTARA': 'SOTARA ',
                   'TUMACO': 'SAN ANDRES DE TUMACO'
                  }

# Aplicar las correcciones a la columna de municipios en la tabla df_dim_sede_ips
df_capacidad_instalada['Municipio'] = df_capacidad_instalada['Municipio'].replace(valores_cambiar)

In [24]:
# revision NOMBRE_MUN con Municipio 

municipios_divipola = set(df_dim_divipola['NOMBRE_MUN'].unique())
municipios_ips = set(df_capacidad_instalada['Municipio'].unique())

print("Municipios en DIVIPOLA pero NO en IPS:")
print(municipios_divipola - municipios_ips)
print("\nMunicipios en IPS pero NO en DIVIPOLA:")
print(municipios_ips - municipios_divipola)
print(f"\nTotal municipios que coinciden: {len(municipios_divipola & municipios_ips)}")

Municipios en DIVIPOLA pero NO en IPS:
{'PAPUNAHUA', 'PACOA', 'SOTARA PAISPAMBA', 'MORICHAL', 'PANA PANA', 'YAVARATE', 'MANAURE BALCON DEL CESAR', 'SAN ANDRES DE CUERQUIA', 'SAN PEDRO DE LOS MILAGROS', 'LA GUADALUPE', nan, 'NUEVO BELEN DE BAJIRA', 'CACAHUAL'}

Municipios en IPS pero NO en DIVIPOLA:
set()

Total municipios que coinciden: 1025


In [25]:
# merge entre df_dim_sede_ips y df_dim_divipola para traer COD_MUN, COD_DPTO tomando como llave Departamento y Municipio
df_dim_sede_ips = df_capacidad_instalada.merge(df_dim_divipola[['COD_MUN', 'COD_DPTO', 'NOMBRE_MUN', 'NOMBRE_DPTO']],
                                        left_on=['Departamento', 'Municipio'],
                                        right_on=['NOMBRE_DPTO', 'NOMBRE_MUN'],
                                        how='left')
df_dim_sede_ips.drop(columns=['NOMBRE_MUN', 'NOMBRE_DPTO'], inplace=True)

In [26]:
df_dim_sede_ips.head()

,Departamento,Municipio,Código prestador,Nombre prestador,nit IPS,num digito_verificion,naturaleza,num nivel atencion,Código sede,Número sede,...,Dirección,Email,Teléfono,nom grupo capacidad,nom descripcion capacidad,num cantidad capacidad instalada,Fecha Corte,Fuente,COD_MUN,COD_DPTO
0,ANTIOQUIA,APARTADO,504512253,URAMEDICOS S.A.S.,"900,497,151",1.0,PRIVADA,NaN,504512253,1,...,KR 98 # 103-29/37,uramedicos@gmail.com,8286090,SALAS,PROCEDIMIENTOS,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...,05045,05
1,ANTIOQUIA,APARTADO,504512253,URAMEDICOS S.A.S.,"900,497,151",1.0,PRIVADA,NaN,504512253,1,...,KR 98 # 103-29/37,uramedicos@gmail.com,8286090,SALAS,PROCEDIMIENTOS,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...,05045,05
2,ANTIOQUIA,APARTADO,504512253,URAMEDICOS S.A.S.,"900,497,151",1.0,PRIVADA,NaN,504512253,1,...,KR 98 # 103-29/37,uramedicos@gmail.com,8286090,SALAS,PROCEDIMIENTOS,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...,05045,05
3,ANTIOQUIA,APARTADO,504512253,URAMEDICOS S.A.S.,"900,497,151",1.0,PRIVADA,NaN,504512253,1,...,KR 98 # 103-29/37,uramedicos@gmail.com,8286090,SALAS,PROCEDIMIENTOS,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...,05045,05
4,ANTIOQUIA,APARTADO,504512253,URAMEDICOS S.A.S.,"900,497,151",1.0,PRIVADA,NaN,504512253,1,...,KR 98 # 103-29/37,uramedicos@gmail.com,8286090,SALAS,PROCEDIMIENTOS,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...,05045,05


### Analisis Exploratorio Registro de Prestadores de Salud (tabla 3)

In [29]:
#lectura de la tabla registro_prestadores_salud
df_prestadores_salud = pd.read_excel(
    config["sources"]["registro_prestadores_salud"]["path"],
    sheet_name=config["sources"]["registro_prestadores_salud"]["sheetname"]
)

#convertir todas las columnas a string
df_prestadores_salud = df_prestadores_salud.astype(str)

print(df_prestadores_salud)



      CodigoPrestador                        NombrePrestador  \
0          9100100017            MANUEL DE JESUS LUBO OROZCO   
1          9100100019  E.S.E. HOSPITAL SAN RAFAEL DE LETICIA   
2          9100100019  E.S.E. HOSPITAL SAN RAFAEL DE LETICIA   
3          9100100019  E.S.E. HOSPITAL SAN RAFAEL DE LETICIA   
4          9100100019  E.S.E. HOSPITAL SAN RAFAEL DE LETICIA   
...               ...                                    ...   
76816      9952400163            JES√öS ALBERTO FONTALVO RUA   
76817      9962400150                jose omar ospina jaimes   
76818      9977300130                    IPS JIWISALUD S.A.S   
76819      9977300145         IPS INDIGENA MATAVEN SALUD SAS   
76820      9977300151             Elkin Ovelio Garc√≠a Valor   

      CodigoHabilitacionSede  \
0               910010001701   
1               910010001901   
2               912630001911   
3               914050001914   
4               914070001915   
...                      ...   
76816  

In [34]:
#lectura de tabla tiempo
df_timepo = pd.read_csv(
    config["sources"]["tiempo"]["path"], sep=';', encoding="utf-8", dtype=str
)
print(df_timepo)

      idtiempo                    fecha periodo mes  nombre_mes  \
0     20291113  2029-11-13 00:00:00,000    2029  11   NOVIEMBRE   
1     20291120  2029-11-20 00:00:00,000    2029  11   NOVIEMBRE   
2     20291127  2029-11-27 00:00:00,000    2029  11   NOVIEMBRE   
3     20291204  2029-12-04 00:00:00,000    2029  12   DICIEMBRE   
4     20291211  2029-12-11 00:00:00,000    2029  12   DICIEMBRE   
...        ...                      ...     ...  ..         ...   
3648  20290902  2029-09-02 00:00:00,000    2029   9  SEPTIEMBRE   
3649  20290909  2029-09-09 00:00:00,000    2029   9  SEPTIEMBRE   
3650  20290916  2029-09-16 00:00:00,000    2029   9  SEPTIEMBRE   
3651  20290923  2029-09-23 00:00:00,000    2029   9  SEPTIEMBRE   
3652  20290930  2029-09-30 00:00:00,000    2029   9  SEPTIEMBRE   

     nombre_mes_corto dia dia_semana nombre_dia_semana es_fin_de_semana  \
0                 NOV  13          2            MARTES             NFDS   
1                 NOV  20          2         